In [24]:
import pandas as pd
import numpy as np

In [25]:
# Task 1: Load Titanic Dataset
df = pd.read_csv('titanic-data-clean.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000000,1,0,A/5 21171,7.2500,UNK,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000000,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000000,0,0,STON/O2. 3101282,7.9250,UNK,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000000,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000000,0,0,373450,8.0500,UNK,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.000000,0,0,211536,13.0000,UNK,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.000000,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,29.699118,1,2,W./C. 6607,23.4500,UNK,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.000000,0,0,111369,30.0000,C148,C


In [26]:
# Task 2: GroupBy
## Task 2.1: Compute the mean of the Fare of passengers from each Pclass
result2_1 = df.groupby('Pclass')['Fare'].mean()
print(result2_1)

Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64


In [27]:
## Task 2.2: Compute the standard deviation of the Fare of passengers from each Pclass and Sex
result2_2 = df.groupby(['Pclass','Sex'])['Fare'].std()
print(result2_2)

Pclass  Sex   
1       female    74.259988
        male      77.548021
2       female    10.891796
        male      14.922235
3       female    11.690314
        male      11.681696
Name: Fare, dtype: float64


In [28]:
## Task 2.3: Compute the mean of the Fare of passengers from each Pclass and Age_group
bins = [0, 25, 35, 60, 100]
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

df['Age_group'] = pd.cut(df['Age'], bins=bins, labels=group_names, right=True)
print(df['Age_group'])

0           Youth
1      MiddleAged
2      YoungAdult
3      YoungAdult
4      YoungAdult
          ...    
886    YoungAdult
887         Youth
888    YoungAdult
889    YoungAdult
890    YoungAdult
Name: Age_group, Length: 891, dtype: category
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']


In [29]:
result2_3 = df.groupby(['Pclass','Age_group'])['Fare'].mean()
print(result2_3)

Pclass  Age_group 
1       Youth         116.987200
        YoungAdult     78.512619
        MiddleAged     76.983334
        Senior         59.969050
2       Youth          25.102309
        YoungAdult     17.535386
        MiddleAged     19.760638
        Senior         10.500000
3       Youth          13.865335
        YoungAdult     13.727935
        MiddleAged     13.334195
        Senior          7.820000
Name: Fare, dtype: float64


C:\Users\Acer\AppData\Local\Temp\ipykernel_463980\2667944672.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result2_3 = df.groupby(['Pclass','Age_group'])['Fare'].mean()


In [30]:
## Task 2.4: Create a Custom Aggregation Function
def gt_mean(values):
    return np.sum(values > np.mean(values))

result2_4 = df.groupby(['Embarked','Sex'])['Fare'].agg(Fare=gt_mean)
print(result2_4)

                 Fare
Embarked Sex         
C        female    30
         male      25
Q        female     8
         male      12
S        female    53
         male     137
UNK      female     0


In [31]:
# Task 3: Multiple Aggregation Functions
## Task 3.1: Compute the mean and standard deviation of the Fare of passengers from each Pclass and Sex
result3_1 = df.groupby(['Pclass','Sex']).agg({
    "Fare": ['mean', 'std']
    })
print(result3_1)

                     Fare           
                     mean        std
Pclass Sex                          
1      female  106.125798  74.259988
       male     67.226127  77.548021
2      female   21.970121  10.891796
       male     19.741782  14.922235
3      female   16.118810  11.690314
       male     12.661633  11.681696


In [32]:
## Task 3.2: Use different aggregation functions on different columns
result3_2 = df.groupby(["Pclass", "Sex"]).agg({
    "Fare": ["min", "max", "mean", "std"], 
    "Age": gt_mean,
    "Parch": "count"              
})
print(result3_2)

                  Fare                                      Age Parch
                   min       max        mean        std gt_mean count
Pclass Sex                                                           
1      female  25.9292  512.3292  106.125798  74.259988      44    94
       male     0.0000  512.3292   67.226127  77.548021      53   122
2      female  10.5000   65.0000   21.970121  10.891796      38    76
       male     0.0000   73.5000   19.741782  14.922235      47   108
3      female   6.7500   69.5500   16.118810  11.690314      81   144
       male     0.0000   69.5500   12.661633  11.681696     202   347


In [33]:
## Task 3.3: A Summary Table with Named Columns
def fare_range(values):
    return values.max() - values.min()
result3_3 = df.groupby("Pclass").agg(
    passengers=("Pclass", "count"),
    average_fare=("Fare", "mean"),
    fare_range=("Fare", fare_range)
    )
print(result3_3)

        passengers  average_fare  fare_range
Pclass                                      
1              216     84.154687    512.3292
2              184     20.662183     73.5000
3              491     13.675550     69.5500


In [34]:
# Task 4: Top Fares Within Each Class
result4_1 = df.sort_values(by=['Pclass','Fare'], ascending=[True, False])
print(result4_1[["PassengerId", "Sex", "Fare", "Age_group"]])

     PassengerId     Sex      Fare   Age_group
258          259  female  512.3292  YoungAdult
679          680    male  512.3292  MiddleAged
737          738    male  512.3292  YoungAdult
27            28    male  263.0000       Youth
88            89  female  263.0000       Youth
..           ...     ...       ...         ...
378          379    male    4.0125       Youth
179          180    male    0.0000  MiddleAged
271          272    male    0.0000       Youth
302          303    male    0.0000       Youth
597          598    male    0.0000  MiddleAged

[891 rows x 4 columns]


In [35]:
# Task 5: Pivot Table
## Task 5.1: Create a pivot table to aggregate the Fare, Parch, and SibSp of passengers in each Pclass group using mean
result5_1 = pd.pivot_table(
    df,
    values=['Fare', 'Parch', 'SibSp'],
    index='Pclass',
    aggfunc='mean'
)
print(result5_1)

             Fare     Parch     SibSp
Pclass                               
1       84.154687  0.356481  0.416667
2       20.662183  0.380435  0.402174
3       13.675550  0.393075  0.615071


In [36]:
## Task 5.2: Extend the code from Task 5.1 to aggregate using count and mean
result5_2 = pd.pivot_table(
    df,
    values=['Fare', 'Parch', 'SibSp'],
    index='Pclass',
    aggfunc=['count','mean']
)
print(result5_2)

       count                   mean                    
        Fare Parch SibSp       Fare     Parch     SibSp
Pclass                                                 
1        216   216   216  84.154687  0.356481  0.416667
2        184   184   184  20.662183  0.380435  0.402174
3        491   491   491  13.675550  0.393075  0.615071


In [37]:
## Task 5.3: Extend the code from Task 5.1 to further compute the mean of each sex for Fare, Parch, SibSp
result5_3 = pd.pivot_table(
    df,
    values=['Fare', 'Parch', 'SibSp'],
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
)
print(result5_3)

              Fare                Parch               SibSp          
Sex         female       male    female      male    female      male
Pclass                                                               
1       106.125798  67.226127  0.457447  0.278689  0.553191  0.311475
2        21.970121  19.741782  0.605263  0.222222  0.486842  0.342593
3        16.118810  12.661633  0.798611  0.224784  0.895833  0.498559


In [38]:
## Task 5.4: Extend the code from Task 5.3 to also compute the mean from all passengers
result5_4 = pd.pivot_table(
    df,
    values=['Fare', 'Parch', 'SibSp'],
    index='Pclass',
    columns='Sex',
    aggfunc='mean',
    margins=True,
    margins_name="All"
)
print(result5_4)

              Fare                           Parch                      \
Sex         female       male        All    female      male       All   
Pclass                                                                   
1       106.125798  67.226127  84.154687  0.457447  0.278689  0.356481   
2        21.970121  19.741782  20.662183  0.605263  0.222222  0.380435   
3        16.118810  12.661633  13.675550  0.798611  0.224784  0.393075   
All      44.479818  25.523893  32.204208  0.649682  0.235702  0.381594   

           SibSp                      
Sex       female      male       All  
Pclass                                
1       0.553191  0.311475  0.416667  
2       0.486842  0.342593  0.402174  
3       0.895833  0.498559  0.615071  
All     0.694268  0.429809  0.523008  


In [39]:
## Task 5.5: Different Aggregations and Named Columns
result5_5 = pd.pivot_table(
    df,
    values=['Age', 'Fare'],
    index='Pclass',
    columns="Sex",
    aggfunc={'Age': 'max', 'Fare': 'median'}
)

result5_5 = result5_5.rename(
    columns={"Age": "max_age", "Fare": "median_fare"},
    level=0 
    )

print(result5_5)

       max_age       median_fare         
Sex     female  male      female     male
Pclass                                   
1         63.0  80.0    82.66455  41.2625
2         57.0  70.0    22.00000  13.0000
3         63.0  74.0    12.47500   7.9250


In [40]:
## Task 5.6 (Optional): A Custom Function in a Pivot Table
result5_6 = pd.pivot_table(
    df,
    values='Fare',
    index='Pclass',
    columns="Sex",
    aggfunc=fare_range
)

print(result5_6)

Sex     female      male
Pclass                  
1        486.4  512.3292
2         54.5   73.5000
3         62.8   69.5500


In [41]:
# Task 6: Fill Missing Fares
df2 = df.copy()
drop_idx = df.sample(frac=0.1, random_state=42).index
df2.loc[drop_idx, "Fare"] = np.nan
print(df2.loc[drop_idx, ["PassengerId", "Pclass", "Fare"]])

     PassengerId  Pclass  Fare
709          710       3   NaN
439          440       2   NaN
840          841       3   NaN
720          721       2   NaN
39            40       3   NaN
..           ...     ...   ...
174          175       1   NaN
493          494       1   NaN
215          216       1   NaN
309          310       1   NaN
822          823       1   NaN

[89 rows x 3 columns]


In [42]:
result6 = df2.groupby('Pclass')['Fare'].transform('mean')
df2['Fare'] = df2['Fare'].fillna(result6)
print(df2.loc[drop_idx, ["PassengerId", "Pclass", "Fare"]])

     PassengerId  Pclass       Fare
709          710       3  13.809933
439          440       2  20.327466
840          841       3  13.809933
720          721       2  20.327466
39            40       3  13.809933
..           ...     ...        ...
174          175       1  85.934486
493          494       1  85.934486
215          216       1  85.934486
309          310       1  85.934486
822          823       1  85.934486

[89 rows x 3 columns]


In [43]:
# Task 7: Row-Level Comparison and Group Filtering
## Task 7.1: Compare Each Fare with Its Class Median
result7_1 = df.copy()
fare_vs_class_median = result7_1.groupby('Pclass')['Fare'].transform('median')
result7_1['fare_vs_class_median'] = result7_1['Fare'] - fare_vs_class_median

print(result7_1[["Pclass", "Fare", "fare_vs_class_median"]])

     Pclass     Fare  fare_vs_class_median
0         3   7.2500               -0.8000
1         1  71.2833               10.9958
2         3   7.9250               -0.1250
3         1  53.1000               -7.1875
4         3   8.0500                0.0000
..      ...      ...                   ...
886       2  13.0000               -1.2500
887       1  30.0000              -30.2875
888       3  23.4500               15.4000
889       1  30.0000              -30.2875
890       3   7.7500               -0.3000

[891 rows x 3 columns]


In [44]:
## Task 7.2: Keep Groups with at Least 40 Passengers
result7_2 = df.groupby(['Embarked', 'Sex']).filter(lambda x: len(x) >= 40)

print(result7_2.groupby(['Embarked', 'Sex']).size())
print("\n")
print(len(result7_2))

Embarked  Sex   
C         female     73
          male       95
Q         male       41
S         female    203
          male      441
dtype: int64


853
